In [1]:
import itertools
import pandas as pd

methods = ["BB", "FIU", "FIW"]
noise_types = ["clear", "Gaussian white"]
images = ["squares", "Gaussian", "MNIST"]
grid_sizes_default = [16, 20, 24]
epsilons_default = [0.02, 0.05, 0.1]

design = []

for method, noise, image in itertools.product(methods, noise_types, images):
    
    if image == "MNIST":
        grid_sizes = [28]
    else:
        grid_sizes = grid_sizes_default
    
    if method in ["BB", "FIU"]:
        epsilons = [1]
    else:
        epsilons = epsilons_default
    
    for grid_size, epsilon in itertools.product(grid_sizes, epsilons):
        design.append((method, noise, image, grid_size, epsilon))

df = pd.DataFrame(design, columns=["Method", "Noise", "Image", "GridSize", "Epsilon"])
df.insert(0, "RunID", range(1, len(df) + 1))

df.to_csv("factorial_design.csv", index=False)

print(f"Created factorial_design.csv with {len(df)} runs.")

Created factorial_design.csv with 70 runs.


In [8]:
df = pd.read_csv("factorial_design.csv")

from pathlib import Path
import importlib.util

this_dir = Path.cwd()

fisher_path = this_dir / "Fisher_reg_analysis.py"
fisher_spec = importlib.util.spec_from_file_location("fisher_reg_analysis", fisher_path)
fisher_mod = importlib.util.module_from_spec(fisher_spec)
fisher_spec.loader.exec_module(fisher_mod)

run_fisher = fisher_mod.run_fisher

bb_path = this_dir / "BenamouBrenier_analysis.py"
bb_spec = importlib.util.spec_from_file_location("benamou_brenier_analysis", bb_path)
bb_mod = importlib.util.module_from_spec(bb_spec)
bb_spec.loader.exec_module(bb_mod)

run_bb = bb_mod.run_bb

import csv
from pathlib import Path

output_path = Path("experiment_results.csv")

with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["run_id", "method", "noise", "objective", "runtime", "SSIM"]
    )
    writer.writeheader()

def run_single(row):
    run_id = row["RunID"]
    method = row["Method"]
    noise = row["Noise"]
    image = row["Image"]
    grid_size = row["GridSize"]
    epsilon = row["Epsilon"]

    if method == "BB":
        obj, runtime, SSIM = run_bb(
            run_id=run_id,
            noise_type=noise,
            image_type=image,
            grid_size=grid_size
        )
    elif method in ["FIU", "FIW"]:
        obj, runtime, SSIM = run_fisher(
            run_id=run_id,
            noise_type=noise,
            image_type=image,
            grid_size=grid_size,
            epsilon=epsilon,
            weighted=(method == "FIW")
        )

    return {
        "run_id": run_id,
        "method": method,
        "noise": noise,
        "objective": obj,
        "runtime": runtime,
        "SSIM": SSIM
    }

for _, row in df.iterrows():
    result = run_single(row)

    with open(output_path, "a", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_id", "method", "noise", "objective", "runtime", "SSIM"]
        )
        writer.writerow(result)

    print(f"Completed Run {result['run_id']}")

print("All runs completed and written incrementally to experiment_results.csv.")

sum(p0) = 1.0 sum(p1) = 1.0
0 : alpha:  0.044781477759207766 alpha_pos:  
1 : alpha:  0.03243807580743151 alpha_pos:  
2 : alpha:  0.04148942582647134 alpha_pos:  
3 : alpha:  0.04452361873994271 alpha_pos:  
4 : alpha:  0.05969931436055017 alpha_pos:  
5 : alpha:  0.05637638526890888 alpha_pos:  
6 : alpha:  0.08069436740874188 alpha_pos:  
7 : alpha:  0.04799310294215795 alpha_pos:  
8 : alpha:  0.09087807076894601 alpha_pos:  
9 : alpha:  0.11515683970999864 alpha_pos:  
10 : alpha:  0.3 alpha_pos:  0.3235384694010463
11 : alpha:  0.3 alpha_pos:  0.5030087517307279
12 : alpha:  0.3 alpha_pos:  0.9211842733649018
13 : alpha:  0.3 alpha_pos:  1.0
14 : alpha:  0.3 alpha_pos:  1.0
15 : alpha:  0.3 alpha_pos:  0.9428248684393965
16 : alpha:  0.3 alpha_pos:  1.0
17 : alpha:  0.3 alpha_pos:  1.0
18 : alpha:  0.3 alpha_pos:  1.0
19 : alpha:  0.3 alpha_pos:  1.0
20 : alpha:  0.3 alpha_pos:  1.0
21 : alpha:  0.3 alpha_pos:  1.0
22 : alpha:  0.3 alpha_pos:  1.0
23 : alpha:  0.3 alpha_pos:  1.0

In [10]:
import pandas as pd

df = pd.read_csv("experiment_results.csv")

method_map = {
    "BB": "Benamou-Brenier",
    "FIU": "Unweighted Fisher Information Regularized",
    "FIW": "Weighted Fisher Information Regularized"
}

df["MethodFull"] = df["method"].map(method_map)

stats = []

for method, group in df.groupby("MethodFull"):
    avg_runtime = group["runtime"].mean()
    avg_ssim = group["SSIM"].mean()
    
    avg_ssim_clear = group[group["noise"] == "clear"]["SSIM"].mean()
    avg_ssim_gaussian = group[group["noise"] == "Gaussian white"]["SSIM"].mean()
    
    robustness = avg_ssim_gaussian / avg_ssim_clear if avg_ssim_clear != 0 else float("nan")
    
    stats.append({
        "Method": method,
        "Average Runtime": avg_runtime,
        "Average SSIM": avg_ssim,
        "Average Robustness": robustness
    })

stats_df = pd.DataFrame(stats)

stats_df.to_csv("experiment_stats.csv", index=False)

print("Created experiment_stats.csv")

Created experiment_stats.csv
